This program:
1. Import environment variables using load_api_key function
2. Fetches games IDs from steam database (each game has an unique ID; steam DB has ~162,000 games)
3. Using data IDs fetch each game information from server
4. Structurize server about about the game; get rid of irrelevant attributes
5. saves them all in json files, with limit of 50mb / file.

In [1]:
import json 
import requests
import time
import glob
from dotenv import load_dotenv
import os
import re


In [2]:
def load_api_key():
    """
    Load Steam API keys from the .env file located in the .venv folder.
    
    Returns:
        tuple: A tuple containing (access_token, web_api_key).
    """
    # Notebook is in the 'src/' folder, so go up one level to reach '.venv/.env'
    env_path = "../.venv/.env" 
    
    load_dotenv(dotenv_path=env_path)
    access_token = os.getenv('access_token')
    web_api_key = os.getenv('web_api_key')
    return access_token, web_api_key

access_token, web_api_key = load_api_key()

In [3]:
# This does not need to be executed; it's aready done and saved in directory: /data/games_id.json 
def get_apps_id(access_token):
    """
    Fetch the list of all Steam games and their IDs using pagination.
    
    Returns:
        dict: A combined JSON response containing the list of all apps if successful.
    """
    all_apps = []
    last_appid = 0
    have_more_results = True
    
    while have_more_results:
        # Steam API uses 'key=' instead of 'access_token='
        url = f"https://api.steampowered.com/IStoreService/GetAppList/v1/?key={access_token}&last_appid={last_appid}&max_results=10000"
        
        response = requests.get(url)
        if response.status_code == 200:
            games_data = response.json()
            response_data = games_data.get("response", {})
            
            apps_batch = response_data.get("apps", [])
            all_apps.extend(apps_batch)
            
            have_more_results = response_data.get("have_more_results", False)
            last_appid = response_data.get("last_appid", last_appid)
            
            # Optional: Sleep briefly to avoid hitting rate limits too quickly
            time.sleep(0.5)
        else: 
            print(f"error: {response.status_code}")
            break
            
    # Structure the final combined output similar to the original single response
    final_output = {
        "response": {
            "apps": all_apps,
            "have_more_results": False,
            "last_appid": last_appid
        }
    }
    
    output_path = "../data/games_id_all.json"
    with open(output_path, "w") as f:
        json.dump(final_output, f)
    
    return final_output

# Use web_api_key instead of access_token
games_id = get_apps_id(web_api_key)
print(f"Total games fetched: {len(games_id['response']['apps'])}")

KeyboardInterrupt: 

In [3]:
def get_games_raw_data(app_id):
    """
    Fetch detailed store information for a specific game by its App ID.
    Saves the raw JSON response to a local file and returns the parsed game data.
    
    Args:
        app_id (int or str): The Steam Application ID to fetch data for.
        
    Returns:
        tuple: The game's store data details, game name, and response status code.
    """
    url = f"https://store.steampowered.com/api/appdetails?appids={app_id}"
    response = requests.get(url)
    
    data = None
    game_name = None
    
    if response.status_code == 200:
        try:
            data = response.json()
            str_app_id = str(app_id)
            game_name = data[str_app_id]["data"]["name"]
            output_path = f"../data/games_informations/{game_name}_{app_id}.json"
            # with open(output_path, "w") as f:
            #     json.dump(data, f)
        except KeyError:
            print(f"Skipping {app_id}: No valid store data or name found.")

    else:
        print(f"Failed to fetch {app_id}. Status code: {response.status_code}")
        
    return data, game_name, response.status_code

In [4]:
def structurize_game_output(raw: dict, gamename):
    first_key = list(raw.keys())[0]
    if isinstance(raw.get(first_key), dict) and "data" in raw[first_key]:
        raw_data = raw[first_key]["data"]
    else:
        raw_data = raw
        
    to_pop = ["type", "steam_appid", "required_age", "detailed_description", "short_description",
              "capsule_image", "capsule_imagev5", "website", "pc_requirements","mac_requirements",
              "packages", "package_groups", "linux_requirements", "platforms", "metacritic", 
              "screenshots", "background", "background_raw", "content_descriptors", "ratings",
              "movies", "legal_notice"]        
    for att in to_pop:
        raw_data.pop(att, None)

    to_pop_price_overview = ["initial_formatted", "final_formatted"]
    if "price_overview" in raw_data:
        for att in to_pop_price_overview:
            if att in raw_data["price_overview"]:
                raw_data["price_overview"].pop(att)
    
    if not raw_data.get("achievements"):
        raw_data["achievements"] = None
    
    raw_data["game_link"] = f"https://store.steampowered.com/app/{first_key}/{gamename}/"

    structurized = {}
    structurized[first_key] = raw_data
        
    return structurized

In [ ]:
all_ids = []
with open('../data/games_id_all.json', 'r') as f:
    games_id = json.load(f)
if "response" in games_id and "apps" in games_id["response"]:
    for app in games_id["response"]["apps"]:
        all_ids.append(app["appid"])

output_dir = "../data/games_informations/"
os.makedirs(output_dir, exist_ok=True)

# 1. Read existing attempted IDs (both successful and failed are stored here)
attempted_file = os.path.join(output_dir, "attempted_ids.txt")
existing_app_ids = set()

if os.path.exists(attempted_file):
    with open(attempted_file, "r") as f:
        for line in f:
            line = line.strip()
            # Handle lines with status codes (e.g., "12345 200")
            parts = line.split()
            if parts and parts[0].isdigit():
                existing_app_ids.add(int(parts[0]))

print(f"Found {len(existing_app_ids)} already attempted games.")

# 2. Logic for chunked files
max_lines = 10000
chunk_index = 1

# Find the current highest chunk index
existing_chunks = glob.glob(os.path.join(output_dir, "games_chunk_*.jsonl"))
if existing_chunks:
    indexes = []
    for chunk in existing_chunks:
        try:
            # Extract number from 'games_chunk_X.jsonl'
            idx = int(os.path.basename(chunk).split('_')[2].split('.')[0])
            indexes.append(idx)
        except ValueError:
            pass
    if indexes:
        chunk_index = max(indexes)

current_chunk_file = os.path.join(output_dir, f"games_chunk_{chunk_index}.jsonl")

# 3. Fetch missing ones
for n in all_ids:
    if n in existing_app_ids:
        # Already downloaded or attempted, skip to next without hitting the API
        continue
        
    rawww, gamename, status_code = get_games_raw_data(n)
    time.sleep(1.5)  # Protection limit
    
    if gamename is not None and rawww is not None:
        # Structurize and write one line to the JSONL file
        structurized_data = structurize_game_output(rawww, gamename)
        
        # Check current chunk file lines limit
        if os.path.exists(current_chunk_file):
            with open(current_chunk_file, 'r', encoding="utf-8") as temp_f:
                line_count = sum(1 for line in temp_f)
            if line_count >= max_lines:
                chunk_index += 1
                current_chunk_file = os.path.join(output_dir, f"games_chunk_{chunk_index}.jsonl")
            
        with open(current_chunk_file, "a", encoding="utf-8") as chunk_f:
            chunk_f.write(json.dumps(structurized_data) + "\n")
            
        print(f"Saved JSON for {gamename} (AppID: {n}) in chunk {chunk_index}")
        
        # Append logic added to only write to txt if fetch was successfully done
        existing_app_ids.add(n)
        with open(attempted_file, "a") as f:
            f.write(f"{n} {status_code}\n")
    else:
        print(f"Recorded failed/invalid request for AppID: {n}")

Found 5170 already attempted games.
Skipping 326740: No valid store data or name found.
Skipping 326740: No valid store data or name found.
Recorded failed/invalid request for AppID: 326740
Recorded failed/invalid request for AppID: 326740
Skipping 354830: No valid store data or name found.
Skipping 354830: No valid store data or name found.
Recorded failed/invalid request for AppID: 354830
Recorded failed/invalid request for AppID: 354830
Saved JSON for Uncanny Valley (AppID: 359580) in chunk 1
Saved JSON for Uncanny Valley (AppID: 359580) in chunk 1
Saved JSON for Assassin’s Creed® Chronicles: Russia (AppID: 359600) in chunk 1
Saved JSON for Assassin’s Creed® Chronicles: India (AppID: 359610) in chunk 1
Saved JSON for Call of Cthulhu: Prisoner of Ice (AppID: 359620) in chunk 1
Saved JSON for Independence War® 2: Edge of Chaos (AppID: 359630) in chunk 1
Saved JSON for Assassin’s Creed® Chronicles: Russia (AppID: 359600) in chunk 1
Saved JSON for Assassin’s Creed® Chronicles: India (Ap